In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
import os

import tqdm
import glob

In [3]:
import torch
from torch.utils.data import DataLoader

from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import r2_score
from sktime.performance_metrics.forecasting import (
    MeanAbsoluteScaledError, 
    MeanAbsolutePercentageError, 
    MeanSquaredError, 
    MeanAbsoluteError
)

In [4]:
import helper as hl
import models as ml
import dataset as ds

# Define Global Variables

In [5]:
SR_FREQ = '12H'
device = torch.device('cpu')

In [6]:
cli_args = dict(
    # LSTM PARAMS
    variant = 'light',  # heavy  | light
    data='dundee',      # dundee | porto | boulder | paloalto
    rnn_cell='gru',     # lstm   | gru
    bi=True,
    hidden_size=24, 
    num_layers=1, 
    fc_layers='24', # D3.2
    # DATA PARAMS
    sr_freq=SR_FREQ, 
    min_pts=100,
    # TRAINING PARAMS
    bs=32, 
    length=48, 
    stride=1, 
    n_rounds=50,
)
cli_args['num_clients'] = 8 if cli_args['data'] == 'dundee' else 4

windowing_params = dict(
    length_min=cli_args['min_pts'], 
    length_max=cli_args['length'], 
    stride=cli_args['stride'],
    rnn_feats=[
        'day_sin', 'day_cos',
        'hour_sin', 'hour_cos', 
        'week_sin', 'week_cos', 
        #
        'power_curr_logdelta',
        'power_next_step1_extrap',
        # 
        'power_curr_std', 
        'power_curr_ema',
        #
        f'downtime_scaled',
        #
        'no_of_sessions_scaled',
        'charging_time',
        'power_curr', 
    ],
    # Extra features to include in the Fully Connected (FC) layer (excl. ```building``` and ```model```)
    fc_feats=[
        'power_output_kW', 
    ],
    y_feats=['power_next'], 
)

# Load Dataset

In [7]:
df_evse_demand_v3 = pd.read_pickle(
    os.path.join(
        '../data', 'pkl', 
        f"{cli_args['data']}_data.demand_{cli_args['sr_freq']}_{cli_args['min_pts']}_points.enriched.v4.pickle"
    )
).sort_index()

df_evse_demand_v3.reset_index(level=2, inplace=True)
df_evse_demand_v3.timestamp = df_evse_demand_v3.timestamp.dt.tz_localize(None)
df_evse_demand_v3.set_index('timestamp', append=True, inplace=True)

In [8]:
df_tr_dev_test = pd.concat(
    {
        cluster_id: pd.read_pickle(
            cluster_id_path
        )
        # 
        for cluster_id, cluster_id_path in enumerate(
            sorted(
                glob.glob(
                    f'../data/pkl/{cli_args["data"]}_data.demand_{cli_args["sr_freq"]}_sequences_{len(windowing_params["rnn_feats"])}_rnn_inputs_{len(windowing_params["fc_feats"])}_fc_inputs_{len(windowing_params["y_feats"])}_outputs_length_{windowing_params["length_max"]}_stride_{windowing_params["stride"]}.cluster_*.v4.pickle'
                )
            )
        )
    },
    names=['cluster_id']
)

In [9]:
building_tokens, building_token_lookup = (
    pd.read_pickle(f'../data/pkl/{cli_args["data"]}_data_12H_building_tokens_v4.pkl'), 
    pd.read_pickle(f'../data/pkl/{cli_args["data"]}_data_12H_building_token_lookup_v4.pkl')
)
model_tokens, model_token_lookup = (
    pd.read_pickle(f'../data/pkl/{cli_args["data"]}_data_12H_model_tokens_v4.pkl'), 
    pd.read_pickle(f'../data/pkl/{cli_args["data"]}_data_12H_model_token_lookup_v4.pkl')
)

In [10]:
df_evcs_meta = pd.read_pickle(f'../data/pkl/{cli_args["data"]}_data.metadata.v3.pickle')

# Load Model

In [11]:
building_embeddings = torch.nn.Embedding(len(building_token_lookup), 15) 
model_embeddings = torch.nn.Embedding(len(model_token_lookup), 3) 

In [12]:
model_params = dict(
    location_embeddings=building_embeddings,
    model_embeddings=model_embeddings,
    input_size=len(windowing_params['rnn_feats']),
    scale=None,
    rnn_cell=getattr(torch.nn, cli_args['rnn_cell'].upper()),
    bidirectional=cli_args['bi'],
    num_layers=cli_args['num_layers'],
    hidden_size=cli_args['hidden_size'],
    misc_size=len(windowing_params['fc_feats']),
    fc_layers=[int(i) for i in cli_args['fc_layers'].split(',')],
    output_size=len(windowing_params['y_feats']),
)

In [13]:
fededf_heavy_dirs = {
    'dundee_lstm': 'fededf_ver2025-08-30_13-35-16_dundee_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
    'dundee_bilstm': 'fededf_ver2025-08-30_13-34-38_dundee_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
    'dundee_gru': 'fededf_ver2025-08-30_19-24-16_dundee_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
    'dundee_bigru': 'fededf_ver2025-08-31_04-10-08_dundee_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
    # 
    'porto_lstm': 'fededf_ver2025-08-31_04-11-57_porto_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
    'porto_bilstm': 'fededf_ver2025-08-31_08-08-53_porto_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
    'porto_gru': 'fededf_ver2025-08-31_07-04-20_porto_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
    'porto_bigru': 'fededf_ver2025-08-31_10-41-12_porto_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
    # 
    'boulder_lstm': 'fededf_ver2025-10-11_17-03-19_boulder_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
    'boulder_bilstm': 'fededf_ver2025-10-11_22-31-59_boulder_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
    'boulder_gru': 'fededf_ver2025-10-12_06-29-26_boulder_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
    'boulder_bigru': 'fededf_ver2025-10-12_06-30-59_boulder_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
    # 
    'paloalto_lstm': 'fededf_ver2025-10-12_07-45-54_paloalto_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
    'paloalto_bilstm': 'fededf_ver2025-10-12_07-46-42_paloalto_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
    'paloalto_gru': 'fededf_ver2025-10-12_12-00-18_paloalto_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
    'paloalto_bigru': 'fededf_ver2025-10-13_15-28-16_paloalto_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
}

fededf_light_dirs = {
    'dundee_lstm': 'fededf_ver2025-11-30_09-46-38_dundee_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
    'dundee_bilstm': 'fededf_ver2025-11-30_11-08-24_dundee_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
    'dundee_gru': 'fededf_ver2025-11-30_11-08-33_dundee_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
    'dundee_bigru': 'fededf_ver2025-11-30_12-07-02_dundee_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
}

fededf_dirs = fededf_heavy_dirs if cli_args['variant'] == 'heavy' else fededf_light_dirs
fededf_dirs

{'dundee_lstm': 'fededf_ver2025-11-30_09-46-38_dundee_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
 'dundee_bilstm': 'fededf_ver2025-11-30_11-08-24_dundee_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
 'dundee_gru': 'fededf_ver2025-11-30_11-08-33_dundee_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1',
 'dundee_bigru': 'fededf_ver2025-11-30_12-07-02_dundee_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1'}

In [14]:
MODEL_RNN_TYPE_NAME = f'{"bi" if model_params["bidirectional"] else ""}{cli_args["rnn_cell"].upper()}'
                    
save_path_best = os.path.join(
    '..', 
    'data', 
    'pth', 
    fededf_dirs[f'{cli_args["data"]}_{"bi" if model_params["bidirectional"] else ""}{cli_args["rnn_cell"]}'],
    f'fededf_{cli_args["data"]}.flwr_global.epoch{cli_args["n_rounds"]}.pth' 
)

print(save_path_best)

../data/pth/fededf_ver2025-11-30_12-07-02_dundee_fraction_fit=1.0_fraction_eval=1.0_proximal_mu=0.1/fededf_dundee.flwr_global.epoch50.pth


In [15]:
def load_model(path, model_config, device):
    # # Evaluate Best Model
    checkpoint = torch.load(path, map_location=device)

    fededf_model = ml.EnergyDemandForecasting_v2(
        **model_config
    )
    fededf_model.to(device)

    # Assign each NumPy array to the corresponding layer in the model
    with torch.no_grad():  # Disable gradient tracking to avoid issues during assignment
        for param, np_array in zip(fededf_model.parameters(), checkpoint['parameters']):
            # Convert NumPy array to a torch tensor with the same dtype as model parameters
            param.copy_(torch.tensor(np_array, dtype=param.dtype))

    fededf_model.eval()
    return fededf_model

# Make Predictions

In [16]:
def fededf_model_inference(model, data_loader):
    y_true_oid_nn, y_pred_oid_nn = [], []

    with torch.no_grad():
        for xb, yb, lb, *args in (pbar := tqdm.tqdm(data_loader, leave=False, total=len(data_loader), dynamic_ncols=True)):
            # print(f'{xb.shape=}\t {yb.shape=}\t {lb.shape=}')
            xb, yb = xb.to(device), yb.to(device)    # Model Inference
            args = (arg.to(device) for arg in args)
            y_pred = model(xb.float(), lb, *args).detach()
            
            y_true_oid_nn.append(yb)
            y_pred_oid_nn.append(y_pred)

    y_true_oid_nn, y_pred_oid_nn = np.concatenate(y_true_oid_nn), np.concatenate(y_pred_oid_nn)
    return y_true_oid_nn, y_pred_oid_nn

In [17]:
def fededf_model_inference_table(evcs_dataset_windows_test, df_evcs_meta, y_true_oid_nn, y_pred_oid_nn, t_horizon=0):
    evcs_dataset_windows_y_pred_time_axis = evcs_dataset_windows_test.time_axes.apply(lambda l: l[-1]).values

    lstm_results = pd.concat(
        {
            MODEL_RNN_TYPE_NAME:pd.Series(
                y_pred_oid_nn[:, t_horizon, :].squeeze(),
                index=[
                    evcs_dataset_windows_test.index.get_level_values(0),
                    evcs_dataset_windows_y_pred_time_axis
                ],
            ).rename_axis(['oid', 'timestamp']),
            'power_next':pd.Series(
                y_true_oid_nn[:, t_horizon, :].squeeze(),
                index=[
                    evcs_dataset_windows_test.index.get_level_values(0),
                    evcs_dataset_windows_y_pred_time_axis
                ]
            ).rename_axis(['oid', 'timestamp']),
        }, 
        axis=1
    )

    lstm_results = lstm_results.loc[~lstm_results.index.duplicated(keep='last')].copy() # Drop duplicated entries (in case of overlapping windows)
    lstm_results = lstm_results.unstack('timestamp')

    return lstm_results.groupby('oid', group_keys=False).apply(
        lambda l: l * (df_evcs_meta.loc[l.name, 'power_output_kW'] * (pd.Timedelta(SR_FREQ).total_seconds() / 3600))
    )

In [18]:
identity_function = FunctionTransformer(None) # We do not need a scaler for now, so we use the Identity function...

evcs_dataset_windows_test = df_tr_dev_test.reorder_levels([1,2,0]).xs(3, level=1, drop_level=False).dropna().copy()

test_dataset = ds.EDFDataset_v2(building_tokens, model_tokens, building_token_lookup, model_token_lookup, evcs_dataset_windows_test, scaler=identity_function)
test_loader = DataLoader(test_dataset,  batch_size=1, shuffle=False, collate_fn=test_dataset.pad_collate)

In [19]:
fededf_global_epoch_i = load_model(
    save_path_best,
    model_params,
    device
)

y_true_oid_nn, y_pred_oid_nn = fededf_model_inference(fededf_global_epoch_i, test_loader)
edf_results = fededf_model_inference_table(evcs_dataset_windows_test, df_evcs_meta, y_true_oid_nn, y_pred_oid_nn, t_horizon=0)

/Users/andrewt/Documents/DataStories-UniPi/FedEDF/src/models.py:44: UserWarning: Instantiated instance without standardization. Falling back to identity function...
  warnings.warn("Instantiated instance without standardization. Falling back to identity function...")
/var/folders/p6/kw8t6dgj12v1289kljz8mz4w0000gn/T/ipykernel_32517/3688296761.py:28: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  lambda l: l * (df_evcs_meta.loc[l.name, 'power_output_kW'] * (pd.Timedelta(SR_FREQ).total_seconds() / 3600))


In [20]:
edf_results_final = edf_results.clip(lower=0).stack(level=1, future_stack=True).dropna()
edf_results_final

biGRU  power_next
oid     timestamp                                 
50230.0 2018-10-20 12:00:00   0.000000    0.000000
        2018-10-21 00:00:00   0.114374    0.000000
        2018-10-21 12:00:00   0.000000    0.000000
        2018-10-22 00:00:00   0.336836    0.000000
        2018-10-22 12:00:00   0.132552    0.000000
...                                ...         ...
51550.0 2018-12-03 00:00:00   1.806012    0.000000
        2018-12-03 12:00:00   1.406760   15.341000
        2018-12-04 00:00:00  10.330979   81.820999
        2018-12-04 12:00:00  38.589397   67.715096
        2018-12-05 00:00:00  49.604271  208.469910

[3990 rows x 2 columns]

# Calculate metrics

In [21]:
train_set_lookup = (
    df_tr_dev_test
    .xs(1, level=2)
    .time_axes
    .explode()
    .groupby(level=1)
    .max()
    .apply(
        lambda l: str(
            l.date()
        )
    )
)

available_oids = list(
    set(
        df_tr_dev_test.xs(3, level=2).dropna().index.get_level_values(1).unique()

    ).intersection(
        set(
            df_tr_dev_test.xs(1, level=2).dropna().index.get_level_values(1).unique()
        )
    )
)

df_evse_demand_v3_train_set = df_evse_demand_v3.loc[
    pd.IndexSlice[
        :, 
        available_oids,
        :
    ]
].groupby(
    'oid', 
    group_keys=False
).apply(
    lambda l: l.loc[
        pd.IndexSlice[
            :, 
            :, 
            :train_set_lookup.loc[l.name]
        ]
    ]
).copy()

y_train = df_evse_demand_v3_train_set['power_curr']
oid_indices = df_evse_demand_v3_train_set['power_curr'].sort_index().groupby('oid', observed=False).groups

In [22]:
model_results_metrics = hl.evaluate_predictions(
    edf_results_final.dropna(),
    y_true_name='power_next',
    y_pred_names=[MODEL_RNN_TYPE_NAME],
    eval_funs=[
        ('MASE_pct', MeanAbsoluteScaledError(sp=24), {'y_train':y_train, 'oid_indices':oid_indices}),
        ('SMAPE_pct', MeanAbsolutePercentageError(symmetric=True), {}),
        ('MAAPE_rads', hl.mean_arctangent_absolute_percentage_error, {}),
        ('WAPE_pct', hl.wape, {}),
        ('RMSE_kW', MeanSquaredError(square_root=True), {}),
        ('MAE_kW', MeanAbsoluteError(), {}),
        ('R2', r2_score, {})
    ]
)

In [23]:
model_results_metrics.groupby(level=0, sort=False).describe().T.loc[
    pd.IndexSlice[:, ['mean', '25%', '50%', '75%']], :
]

0_MASE_pct  1_SMAPE_pct  2_MAAPE_rads  3_WAPE_pct  4_RMSE_kW  \
biGRU mean    0.371119     1.654017      1.281772    2.475302   5.940899   
      25%     0.199561     1.488135      1.152831    1.232264   1.355320   
      50%     0.317409     1.782735      1.376790    1.613575   2.792954   
      75%     0.453168     1.894238      1.481045    2.214240   5.350340   

            5_MAE_kW      6_R2  
biGRU mean  2.462424 -0.104497  
      25%   0.464080  0.004063  
      50%   1.296158  0.140997  
      75%   2.674256  0.289112

In [24]:
edf_results_final.dropna().to_pickle(f'../data/pkl/{cli_args["data"]}_data.demand_{cli_args["sr_freq"]}.EVSE__Fed{MODEL_RNN_TYPE_NAME}_{cli_args["variant"]}__model.v4.pickle')
model_results_metrics.to_pickle(f'../data/pkl/{cli_args["data"]}_data.demand_{cli_args["sr_freq"]}.EVSE__Fed{MODEL_RNN_TYPE_NAME}_{cli_args["variant"]}__model.metrics.v4.pickle')